# Rapport de Projet — Ghost CMS avec Thème One Piece & Pipeline CI/CD

**Projet** : CloudBlogProject  
**Dépôt** : https://github.com/ZoaAlex/CloudBlogProject  
**Date** : Avril 2026  
**Stack** : Ghost 5 · Docker · GitHub Actions · EC2 AWS  


## Sommaire

1. [Présentation du projet](#1-présentation-du-projet)
2. [Architecture](#2-architecture)
3. [Infrastructure Docker](#3-infrastructure-docker)
4. [Thème Ghost — Grand Line Theme](#4-thème-ghost--grand-line-theme)
5. [Pipeline CI/CD](#5-pipeline-cicd)
6. [Gestion Git & Branches](#6-gestion-git--branches)
7. [Déploiement sur EC2](#7-déploiement-sur-ec2)
8. [Problèmes rencontrés & Solutions](#8-problèmes-rencontrés--solutions)
9. [Conclusion](#9-conclusion)


---
## 1. Présentation du projet

Ce projet a pour objectif de déployer un **blog Ghost CMS** avec un thème personnalisé inspiré de l'univers **One Piece**, dans un environnement entièrement conteneurisé avec Docker, versionné avec Git et automatisé via GitHub Actions.

### Objectifs
- Conteneuriser Ghost CMS avec Docker Compose
- Développer un thème Ghost custom (`grand-line-theme`)
- Mettre en place une pipeline CI/CD complète (validation, déploiement staging, déploiement production)
- Automatiser le déploiement sur une instance EC2 AWS via SSH


In [ ]:
# Vérification de l'environnement local
import subprocess, sys

checks = {
    'Docker': ['docker', '--version'],
    'Docker Compose': ['docker', 'compose', 'version'],
    'Git': ['git', '--version'],
    'Python': [sys.executable, '--version'],
}

for name, cmd in checks.items():
    try:
        out = subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode().strip()
        print(f'✅ {name}: {out}')
    except Exception as e:
        print(f'❌ {name}: non disponible')


---
## 2. Architecture

```
CloudBlogProject/
├── .github/
│   └── workflows/
│       ├── ci.yml              ← Validation thème (toutes branches)
│       ├── cd-staging.yml      ← Déploiement EC2 staging (develop)
│       └── cd-production.yml   ← Déploiement EC2 production (main)
├── content/
│   ├── themes/
│   │   └── grand-line-theme/   ← Thème One Piece (versionné)
│   ├── data/                   ← ghost.db SQLite (ignoré Git)
│   └── settings/
│       └── routes.yaml         ← Routes Ghost (versionné)
├── docker-compose.yml
├── .gitignore
├── .gitattributes
└── README.md
```

### Branches Git
| Branche | Rôle | Déploiement |
|---|---|---|
| `main` | Code stable, production | EC2 Production |
| `develop` | Intégration, tests | EC2 Staging |
| `bgBranch` / `bg-taf` | Feature branches | — |


---
## 3. Infrastructure Docker

Ghost tourne dans un conteneur unique basé sur `ghost:5-alpine` (Alpine Linux, image légère).
La base de données utilisée est **SQLite** — suffisante pour le développement de thèmes.


In [ ]:
docker_compose = '''
services:
  blog:
    image: ghost:5-alpine
    container_name: ghost
    restart: always
    ports:
      - '2368:2368'
    environment:
      database__client: sqlite3
      database__connection__filename: /var/lib/ghost/content/data/ghost.db
      url: http://localhost:2368
      NODE_ENV: development
      activeTheme: grand-line-theme
    volumes:
      - ./content/themes/grand-line-theme:/var/lib/ghost/content/themes/grand-line-theme
      - ./content/data:/var/lib/ghost/content/data
      - ./content/settings/routes.yaml:/var/lib/ghost/content/settings/routes.yaml
'''
print(docker_compose)


### Volumes montés
| Volume local | Volume conteneur | Rôle |
|---|---|---|
| `./content/themes/grand-line-theme` | `/var/lib/ghost/content/themes/grand-line-theme` | Thème custom (hot-reload) |
| `./content/data` | `/var/lib/ghost/content/data` | Base SQLite persistante |
| `./content/settings/routes.yaml` | `/var/lib/ghost/content/settings/routes.yaml` | Routes Ghost |

### Commandes Docker essentielles
```bash
docker compose up -d        # Démarrer Ghost
docker compose down         # Arrêter et supprimer
docker compose logs -f      # Suivre les logs
docker restart ghost        # Redémarrer le conteneur
```


---
## 4. Thème Ghost — Grand Line Theme

Thème personnalisé inspiré de l'univers **One Piece** d'Eiichiro Oda.

### Structure du thème
```
grand-line-theme/
├── assets/
│   ├── css/screen.css      ← Styles (palette pirate : rouge, or, bleu océan)
│   └── js/main.js          ← Animations scroll, menu mobile
├── partials/
│   ├── navigation.hbs      ← Header sticky avec recherche
│   ├── footer.hbs          ← Footer avec tags dynamiques
│   ├── post-card.hbs       ← Carte article réutilisable
│   └── wave-divider.hbs    ← Séparateur SVG animé
├── default.hbs             ← Layout HTML de base
├── index.hbs               ← Page d'accueil (hero + grille)
├── post.hbs                ← Article individuel
├── page.hbs                ← Page statique
├── tag.hbs                 ← Archive par tag
├── author.hbs              ← Page auteur
├── error-404.hbs           ← Page 404
└── package.json            ← Config Ghost du thème
```


In [ ]:
import json

pkg = {
    'name': 'grand-line',
    'version': '1.0.0',
    'description': 'Ghost theme inspired by One Piece',
    'author': {'email': 'contact@example.com'},
    'engines': {'ghost': '>=5.0.0'},
    'config': {
        'posts_per_page': 8,
        'card_assets': True,
        'image_sizes': {
            'xs': {'width': 300}, 's': {'width': 600},
            'm': {'width': 960}, 'l': {'width': 1200}, 'xl': {'width': 2000}
        }
    }
}
print(json.dumps(pkg, indent=2, ensure_ascii=False))


---
## 5. Pipeline CI/CD

3 fichiers GitHub Actions couvrent l'intégralité du cycle de vie du code.

### Flux global
```
push (toute branche)  →  ci.yml          (validation)
push develop          →  cd-staging.yml  (merge + deploy staging)
push main             →  cd-production.yml (backup + merge + deploy prod)
```


### 5.1 — ci.yml (Intégration Continue)

Déclenché sur **tous les pushs** et **pull requests** vers `main`/`develop`.

| Étape | Action |
|---|---|
| Checkout | Récupère le code |
| Setup Node.js 20 | Prépare l'environnement |
| Install GScan | Outil de validation officiel Ghost |
| Check theme folder | Vérifie que `grand-line-theme` existe |
| Run GScan | Valide la conformité du thème Ghost 5 |
| Validate routes.yaml | Vérifie la syntaxe YAML avec Python |
| Validate docker-compose | `docker compose config --quiet` |


### 5.2 — cd-staging.yml (Déploiement Staging)

Déclenché sur **push vers `develop`**. 3 jobs séquentiels :

```
merge-on-ec2  →  deploy-staging  →  health-check
```

| Job | Action sur EC2 |
|---|---|
| `merge-on-ec2` | `git fetch --all` + `git merge --ff-only origin/develop` |
| `deploy-staging` | `NODE_ENV=development docker compose up -d` |
| `health-check` | `curl` HTTP sur `:2368`, échec si pas 200/301/302 |


### 5.3 — cd-production.yml (Déploiement Production)

Déclenché sur **push vers `main`**. 4 jobs séquentiels :

```
backup  →  merge-on-ec2  →  deploy-production  →  health-check
```

| Job | Action sur EC2 |
|---|---|
| `backup` | Copie horodatée de `ghost.db` et `routes.yaml` |
| `merge-on-ec2` | `git fetch --all` + `git merge --ff-only origin/main` |
| `deploy-production` | `NODE_ENV=production docker compose up -d` |
| `health-check` | `curl` HTTP + `docker logs` si échec |


---
## 6. Gestion Git & Branches

### .gitignore — Ce qui est exclu
```
content/data/        ← Base SQLite (données)
content/logs/        ← Logs Ghost
content/images/      ← Médias uploadés
*.db / *.log         ← Fichiers binaires/logs
.env                 ← Variables d'environnement
```

### .gitattributes — Normalisation des fins de ligne
Tous les fichiers `.hbs`, `.css`, `.js`, `.yml`, `.json` sont forcés en **LF** pour éviter les conflits entre Windows et Linux (EC2).

### Secrets GitHub Actions requis
| Secret | Description |
|---|---|
| `SSH_HOST` | IP publique de l'EC2 |
| `SSH_USER` | Utilisateur SSH (ex: `ubuntu`) |
| `SSH_PRIVATE_KEY` | Clé privée SSH (contenu du `.pem`) |
| `SSH_PORT` | Port SSH (défaut: 22) |
| `STAGING_DEPLOY_PATH` | Chemin du projet sur l'EC2 staging |
| `PROD_DEPLOY_PATH` | Chemin du projet sur l'EC2 production |


---
## 7. Déploiement sur EC2

### Prérequis sur l'instance EC2
```bash
# Installer Docker
sudo apt update && sudo apt install -y docker.io docker-compose-plugin
sudo usermod -aG docker ubuntu

# Cloner le projet
git clone https://github.com/ZoaAlex/CloudBlogProject.git /home/ubuntu/ghost-blog
cd /home/ubuntu/ghost-blog

# Premier démarrage
docker compose up -d
```

### Mécanisme de merge automatique
À chaque push GitHub, la pipeline SSH sur l'EC2 et exécute :
```bash
git fetch --all
git checkout main          # ou develop
git merge --ff-only origin/main
docker compose up -d --remove-orphans
```
Le `--ff-only` garantit qu'aucun merge commit parasite n'est créé sur le serveur.


---
## 8. Problèmes rencontrés & Solutions

| Problème | Cause | Solution |
|---|---|---|
| `ThemeValidationError` au démarrage Ghost | Helpers invalides dans les templates | Correction `error.hbs` (`eq` → `is404`), `header.hbs` (`@page.isHome` supprimé) |
| 404 sur la navigation | Liens hardcodés vers tags inexistants dans `footer.hbs` | Remplacé par `{{#get "tags"}}` dynamique |
| 404 sur articles liés | Filtre `{{tags.[0].slug}}` invalide dans `post.hbs` | Remplacé par `filter="id:-{{id}}"` |
| `meta_description` déprécié | Changement API Ghost 5 | Supprimé de `default.hbs` (géré par `{{ghost_head}}`) |
| Classes Koenig manquantes | Éditeur Ghost requiert `.kg-width-wide/full` | Ajout dans `style.css` |
| Conflits Git dans `.gitignore` | Merge de branches divergentes | Résolution manuelle des marqueurs `<<<<<<<` |
| `content/settings/` ignoré par Git | `.gitignore` trop large | Ajout de `!content/settings/routes.yaml` |
| CRLF warnings sur Windows | Fins de ligne Windows vs Linux | Ajout `.gitattributes` avec `eol=lf` |


---
## 9. Conclusion

Ce projet met en place une chaîne complète de développement et déploiement pour un blog Ghost CMS :

- **Conteneurisation** : Ghost 5 sur Alpine avec SQLite, volumes Docker pour le hot-reload du thème
- **Thème custom** : `grand-line-theme` inspiré One Piece, validé par GScan, conforme Ghost 5
- **CI/CD** : 3 pipelines GitHub Actions couvrant validation, staging et production
- **Déploiement EC2** : merge automatique via SSH à chaque push sur `develop` ou `main`
- **Versioning propre** : `.gitignore` et `.gitattributes` adaptés, `routes.yaml` versionné

### Améliorations possibles
- Passer de SQLite à MySQL pour la production
- Ajouter un reverse proxy Nginx avec HTTPS (Let's Encrypt)
- Mettre en place des tests de régression visuelle sur le thème
- Ajouter un job de rollback automatique en cas d'échec du health check


In [ ]:
# Résumé des fichiers versionnés
import os

versioned = [
    'docker-compose.yml',
    '.gitignore',
    '.gitattributes',
    'content/settings/routes.yaml',
    '.github/workflows/ci.yml',
    '.github/workflows/cd-staging.yml',
    '.github/workflows/cd-production.yml',
]

print('Fichiers clés du projet :')
for f in versioned:
    exists = os.path.exists(f)
    status = '✅' if exists else '❌'
    size = f'{os.path.getsize(f)} octets' if exists else 'absent'
    print(f'  {status} {f} ({size})')
